# C3 exp10 on v3: the 7B's letter prior (PriDe)

Qwen2.5-7B-Instruct (4-bit), k = 0, definitions on, **data v3**. Does the model prefer some answer letters whatever option they stand for, and does dividing that preference out help? Method: PriDe (Zheng et al. 2024, "Large Language Models Are Not Robust Multiple Choice Selectors", ICLR 2024).

Two settings, each scored under the **7 rotations of the options** (`--option-shift 0..6`: the letters stay A–G in order, the option texts move up by the shift, the examples answer with their option's new letter):
- **shots2**: exp08's best prompt (2 examples per label, seed 0); shift 0 should repeat its 0.341;
- **zs**: zero-shot, exp03's 7B setting; shift 0 should repeat its 0.206 (v3).

Then, without the model (`scripts/pride_prompting.py`): the letter prior, **PriDe** (prior from 5 % of the segments, no labels, 5 draws) and **cyclic** (every segment averaged over its 7 rotations).

Speed: `--prefix-cache`, batch 32, both GPUs. The 14 runs need the cache (without it, ~12 h on both GPUs): if the check in section 5 fails, the notebook stops.

Note: `tasks/C3-prompting-baseline/experiments/exp10_pride-7b/NOTE.md`. Temporary trials (`scripts/trial_prompting.py`); not reportable (PROTOCOL §4.1).

Before running, in the notebook settings: **GPU T4 x2**, **Internet on**, the `smi-v3-meetings` dataset attached (only that one: the data cell looks for v3).

Expected time: about 2 hours (an estimate; the first progress lines of each log show the real speed).

## 1. GPU

In [ ]:
!nvidia-smi

## 2. Code and packages
`bitsandbytes` is needed for the 4-bit 7B model. Do **not** install `requirements.txt` here.

In [ ]:
!rm -rf /kaggle/working/repo
!git clone -b C3/prompting-baseline https://github.com/ahmedwaleedaref/Meeting-Intelligent-System-.git /kaggle/working/repo
%cd /kaggle/working/repo
!git log --oneline -3
!pip install -q transformers==5.20.1 bitsandbytes

## 3. Data folder: the v3 one

In [ ]:
import glob, json, os, subprocess, time

hits = [h for h in glob.glob("/kaggle/input/**/source.json", recursive=True) if json.load(open(h))["data_version"] == "v3"]
assert len(hits) == 1, f"expected exactly one v3 dataset under /kaggle/input, found {hits} : is smi-v3-meetings attached?"
DATA_ROOT = os.path.dirname(hits[0])
print("DATA_ROOT =", DATA_ROOT, "|", json.load(open(hits[0])))
os.makedirs("/kaggle/working/logs", exist_ok=True)

## 4. Download the model once
Both GPUs load the same weights: downloading them here first means neither run waits on the other's download.

In [ ]:
from huggingface_hub import snapshot_download

SEVEN_B = "Qwen/Qwen2.5-7B-Instruct"
print(snapshot_download(SEVEN_B))

## 5. Check the prefix cache on the 7B (~3 min)
16 val segments with exp08's 2 examples per label and the options rotated by 3: the 8 longest are scored with the cache, as a padded batch, and one at a time (the reference). Each run repeats the check on its own longest prompts.

In [ ]:
import shutil

shutil.rmtree("/kaggle/working/checks", ignore_errors=True)  # a throwaway check : always measured afresh
check = subprocess.run(
    ["python", "scripts/trial_prompting.py", "--data-root", DATA_ROOT, "--out-dir", "/kaggle/working/checks",
     "--run-name", "prefix_cache_check", "--model", SEVEN_B, "--load-in-4bit", "--prefix-cache",
     "--shots-per-class", "2", "--shot-seed", "0", "--option-shift", "3", "--limit", "16", "--batch-size", "16"],
    cwd="/kaggle/working/repo", env=dict(os.environ, CUDA_VISIBLE_DEVICES="0", PYTHONPATH="src"),
    capture_output=True, text=True)
print("\n".join(l for l in check.stdout.splitlines() if l.startswith("prefix cache")))
if check.returncode != 0:
    print("\n".join(check.stderr.strip().splitlines()[-3:]))
    raise RuntimeError("the prefix cache check failed : the 14 runs would take ~12 h without it , so the notebook stops here")
SPEED_FLAGS = ["--prefix-cache", "--batch-size", "32"]

## 6. Helpers: a queue of runs on one GPU, and waiting while showing progress

In [ ]:
def queue(gpu, jobs):
    """run the jobs one after another on one GPU , in the background ; each run logs to logs/<run_name>.log"""
    commands = []
    for run_name, extra in jobs:
        commands.append(
            f"python scripts/trial_prompting.py --data-root '{DATA_ROOT}' --out-dir /kaggle/working/runs "
            f"--run-name {run_name} --model {SEVEN_B} --load-in-4bit {' '.join(SPEED_FLAGS + extra)} "
            f"> /kaggle/working/logs/{run_name}.log 2>&1"
        )
    env = dict(os.environ, CUDA_VISIBLE_DEVICES=str(gpu), PYTHONPATH="src")
    # "; " : a failed run does not stop the next one in the queue
    return subprocess.Popen("; ".join(commands), shell=True, cwd="/kaggle/working/repo", env=env)

def last_line(path):
    lines = [l for l in open(path).read().splitlines() if l.strip()]
    return lines[-1] if lines else "(starting)"

def wait(processes):
    """every 2 minutes , print the latest line of every log , until all queues have finished"""
    while any(p.poll() is None for p in processes):
        time.sleep(120)
        for path in sorted(glob.glob("/kaggle/working/logs/*.log")):
            print(f"{os.path.basename(path)[:-4]:26} {last_line(path)[-80:]}")
        print()
    print("all queues finished")

## 7. Run both GPUs (~2 h)
14 runs: each setting under shifts 0–6. Each GPU starts with one setting's shift 0 (the repeat of its baseline) and gets about the same work: shots2 runs take ~15 min, zero-shot ones less.
If the session stops, run this cell again: finished runs are skipped and an interrupted run continues where it stopped.

In [ ]:
SETTINGS = {"shots2": ["--shots-per-class", "2", "--shot-seed", "0"], "zs": []}

def job(setting, shift):
    return (f"c3_exp10_7b_{setting}_r{shift}", SETTINGS[setting] + ["--option-shift", str(shift)])

wait([
    queue(0, [job("shots2", s) for s in range(0, 4)] + [job("zs", s) for s in range(4, 7)]),
    queue(1, [job("zs", s) for s in range(0, 4)] + [job("shots2", s) for s in range(4, 7)]),
])

## 8. PriDe and cyclic averaging (no GPU, ~1 min)
For each setting: the letter prior over all segments, `<setting>_cyclic` (average of the 7 rotations) and `<setting>_pride0..4` (prior from 5 % of the segments, 5 draws). The outputs are never overwritten: to run this cell again, delete them first.

In [ ]:
for setting in SETTINGS:
    runs = [f"/kaggle/working/runs/c3_exp10_7b_{setting}_r{s}" for s in range(7)]
    missing = [r for r in runs if not os.path.exists(f"{r}/metrics.json")]
    if missing:
        print(f"{setting}: skipped , runs missing : {missing}"); continue
    out = subprocess.run(
        ["python", "scripts/pride_prompting.py", "--data-root", DATA_ROOT,
         "--name", f"/kaggle/working/runs/c3_exp10_7b_{setting}", "--run", *runs],
        cwd="/kaggle/working/repo", env=dict(os.environ, PYTHONPATH="src"), capture_output=True, text=True)
    print(f"===== {setting}\n{out.stdout}{out.stderr[-2000:] if out.returncode else ''}")

## 9. Results
Baselines (v3): shots2 0.341 (exp08), zero-shot 0.206 (exp03). Prompt noise (exp07): differences below ~0.035 are not real.

In [ ]:
def show(name):
    path = f"/kaggle/working/runs/{name}"
    if not os.path.exists(f"{path}/metrics.json"):
        print(f"{name:30} (missing : see logs/{name}.log)"); return
    m = json.load(open(f"{path}/metrics.json"))
    predicted_other = sum(json.loads(line)["label"] == "other" for line in open(f"{path}/predictions.jsonl"))
    print(f"{name:30} {m['primary']['macro_f1_6']:8.4f} {m['per_class']['other']['f1']:8.3f} {predicted_other:11,}")

print(f"{'run':30} {'macro-F1':>8} {'other F1':>8} {'pred. other':>11}  (gold other 13,634)")
for setting in SETTINGS:
    for name in [f"c3_exp10_7b_{setting}_r{s}" for s in range(7)] + [f"c3_exp10_7b_{setting}_cyclic"] + [f"c3_exp10_7b_{setting}_pride{d}" for d in range(5)]:
        show(name)
    print()

## 10. Download the outputs
Right-hand panel: **Output** → `/kaggle/working` → `c3_exp10_runs.zip` → **⋮** → **Download**.

In [ ]:
%cd /kaggle/working
!zip -qr c3_exp10_runs.zip runs logs
!ls -la c3_exp10_runs.zip